# Step 09 — Interactive web map

Composes the pipeline's canonical products into a **self-contained, GitHub-Pages-ready
static site** under `outputs/09_webmap/`. It opens locally by double-clicking
`index.html` and deploys to GitHub Pages unchanged.

Layers: tiled **orthomosaic** (both zones), **canopy height (CHM)**, **habitat
classes**, **detected trees** (talar), and the hydrology / **flood** layers
(inundation probability, depression storage) — plus the embedded **interactive 3D
flood-stage scene** and rising-flood animation from Step 08b.

Logic lives in `drone_reserve.webmap`; this notebook only orchestrates. Outputs are
deterministic. Re-tiling the ortho is the only slow part (~3 min) and is skipped if
tiles already exist (set `REBUILD_TILES=True` to force).

In [1]:
from pathlib import Path
import json, shutil

import geopandas as gpd
import pandas as pd

from drone_reserve import webmap as wm

# Resolve the repo root (same pattern as the other notebooks).
for parent in [Path.cwd(), *Path.cwd().parents]:
    if (parent / "pyproject.toml").is_file():
        REPO = parent; break
else:
    raise RuntimeError("repo root not found")

DATA    = REPO / "data"
OUTROOT = REPO / "outputs"
SITE    = OUTROOT / "09_webmap"
for sub in ("tiles", "overlays", "data", "assets"):
    (SITE / sub).mkdir(parents=True, exist_ok=True)

REBUILD_TILES = False  # set True to regenerate the ortho tile pyramids
print("repo:", REPO)
print("site:", SITE)

repo: C:\Users\paco_\OneDrive\Escritorio\repos\drone_reserve
site: C:\Users\paco_\OneDrive\Escritorio\repos\drone_reserve\outputs\09_webmap


## 1. Orthomosaic to XYZ tile pyramid

The orthos are multi-hundred-MB at ~3.4 cm GSD, so we **tile** rather than inline
them (project rule: never load full rasters into RAM). `gdal2tiles` reprojects to
web-mercator and writes slippy-map `{z}/{x}/{y}.png`. The 12-13 ha extents keep each
pyramid small (~70 MB).

In [2]:
ORTHOS = {"talar": DATA / "talar_ortomosaico.tif",
          "pastizal": DATA / "pastizal_ortomosaico.tif"}

for zone, src in ORTHOS.items():
    tdir = SITE / "tiles" / f"{zone}_ortho"
    if not REBUILD_TILES and any(tdir.glob("*/")):
        print(f"[tiles] {zone}: present, skip ({sum(1 for _ in tdir.rglob('*.png'))} tiles)")
        continue
    wm.tile_orthomosaic(src, tdir, zoom="14-21")
    print(f"[tiles] {zone}: {sum(1 for _ in tdir.rglob('*.png'))} tiles")

[tiles] talar: present, skip (980 tiles)
[tiles] pastizal: present, skip (1133 tiles)


## 2. Thematic overlays (CHM, habitat, hydrology)

Each small derived raster is reprojected to EPSG:3857 and colorized to an RGBA PNG,
placed on the map with its 3857 extent converted to 4326 corner bounds — geometrically
exact on a web-mercator map. The two zones share a single CHM scale so they compare
directly; depression-storage colour is capped at 2 m so the few deep ditch/artifact
cells (to ~10 m, flagged in Step 08) don't wash out the shallow wetland-ponding signal.

In [3]:
CHM_VMAX     = 17.0  # shared canopy-height colour scale (m), both zones
STORAGE_VMAX = 2.0   # cap storage colour (m); deeper ditch/artifact cells saturate

continuous = [
    ("talar",    "04_chm/talar_chm_corrected_0p5m.tif", "chm",     "viridis", 0, CHM_VMAX),
    ("pastizal", "04_chm/pastizal_chm_0p5m.tif",        "chm",     "viridis", 0, CHM_VMAX),
    ("talar",    "08_hydro/talar_inund.tif",            "inund",   "Blues",   0, 1),
    ("pastizal", "08_hydro/pastizal_inund.tif",         "inund",   "Blues",   0, 1),
    ("talar",    "08_hydro/talar_storage.tif",          "storage", "YlGnBu",  0, STORAGE_VMAX),
    ("pastizal", "08_hydro/pastizal_storage.tif",       "storage", "YlGnBu",  0, STORAGE_VMAX),
]
meta = {}
for zone, rel, nm, cmap, vmin, vmax in continuous:
    r = wm.make_overlay(OUTROOT / rel, SITE / "overlays" / f"{zone}_{nm}.png",
                        kind="continuous", cmap=cmap, vmin=vmin, vmax=vmax)
    meta[f"{zone}_{nm}"] = {"bounds": r["bounds"], "range": r["range"]}

for zone, rel in [("talar",    "05_segmentation/talar_habitat_0p5m.tif"),
                  ("pastizal", "05_segmentation/pastizal_habitat_0p5m.tif")]:
    r = wm.make_overlay(OUTROOT / rel, SITE / "overlays" / f"{zone}_habitat.png",
                        kind="categorical")
    meta[f"{zone}_habitat"] = {"bounds": r["bounds"]}

(SITE / "overlay_meta.json").write_text(json.dumps(meta, indent=2))
print("overlays:", list(meta))

overlays: ['talar_chm', 'pastizal_chm', 'talar_inund', 'pastizal_inund', 'talar_storage', 'pastizal_storage', 'talar_habitat', 'pastizal_habitat']


## 3. Vectors to GeoJSON (WGS84)

In [4]:
wm.vector_to_geojson(OUTROOT / "06_trees/talar_treetops.gpkg",
                     SITE / "data" / "talar_trees.geojson", columns=["height"])
wm.vector_to_geojson(DATA / "Area.shp", SITE / "data" / "zones.geojson")
print("geojson written")

geojson written


## 4. Copy the Step 08b 3D flood assets into the site

The interactive flood-stage scene and the rising-flood GIF are produced in Step 08b;
we copy them under `assets/` so the site is self-contained.

In [5]:
shutil.copy2(OUTROOT / "08b_3d/talar_inundation_3d.html", SITE / "assets" / "talar_inundation_3d.html")
shutil.copy2(OUTROOT / "08b_3d/talar_flood_rising.gif",   SITE / "assets" / "talar_flood_rising.gif")
print("copied 3D flood html + gif")

copied 3D flood html + gif


## 5. Assemble the interactive map (`map.html`)

Satellite + street basemaps, both orthos on by default, thematic overlays toggleable,
detected trees sized/coloured by height, a single collapsible legend, and the
rising-flood GIF floated as a teaser into the 3D scene.

In [6]:
# Map extent = union of both flight footprints.
zones = gpd.read_file(SITE / "data" / "zones.geojson")
w, s, e, n = zones.total_bounds
center = ((s + n) / 2, (w + e) / 2)
fit = [[s, w], [n, e]]

m = wm.new_map(center, zoom_start=16)
wm.add_tiles(m, "Orthomosaic - talar",    "tiles/talar_ortho",    show=True)
wm.add_tiles(m, "Orthomosaic - pastizal", "tiles/pastizal_ortho", show=True)

def _ov(name, key, opacity=0.85, show=False):
    wm.add_image_overlay(m, name, f"overlays/{key}.png", meta[key]["bounds"],
                         opacity=opacity, show=show)

_ov("Canopy height (CHM) - talar",    "talar_chm")
_ov("Canopy height (CHM) - pastizal", "pastizal_chm")
_ov("Habitat classes - talar",        "talar_habitat")
_ov("Habitat classes - pastizal",     "pastizal_habitat")
_ov("Inundation probability - talar",    "talar_inund",    opacity=0.7)
_ov("Inundation probability - pastizal", "pastizal_inund", opacity=0.7)
_ov("Depression storage - talar",    "talar_storage",    opacity=0.8)
_ov("Depression storage - pastizal", "pastizal_storage", opacity=0.8)

wm.add_geojson_points(m, "Detected trees - talar",
                      str(SITE / "data" / "talar_trees.geojson"),
                      value_field="height", show=False)
wm.add_zone_outlines(m, str(SITE / "data" / "zones.geojson"))

wm.add_legend(m, [
    {"type": "ramp", "label": "Canopy height",          "cmap": "viridis", "vmin": 0, "vmax": CHM_VMAX, "unit": "m"},
    {"type": "ramp", "label": "Inundation probability",  "cmap": "Blues",   "vmin": 0, "vmax": 1, "unit": ""},
    {"type": "ramp", "label": "Depression storage",      "cmap": "YlGnBu",  "vmin": 0, "vmax": STORAGE_VMAX, "unit": "m"},
    {"type": "cats", "label": "Habitat classes",
     "items": [(wm.HABITAT_NAMES[k], wm.HABITAT_COLORS[k]) for k in (0, 1, 2)]},
])
wm.add_float_gif(m, "assets/talar_flood_rising.gif")
wm.finalize(m, fit_bounds=fit)
m.save(str(SITE / "map.html"))
print("map.html:", (SITE / "map.html").stat().st_size // 1024, "KB")

map.html: 1156 KB


## 6. Landing page (`index.html`)

In [7]:
# Headline stats, pulled live from the pipeline's metric tables.
canopy = pd.read_csv(OUTROOT / "07_metrics" / "canopy_metrics.csv").set_index("zone")
hydro  = pd.read_csv(OUTROOT / "08_hydro" / "hydrology_summary.csv")
n_trees = len(gpd.read_file(SITE / "data" / "talar_trees.geojson"))
ponding_pct = 100 * hydro["inundation_area_ha_p50"].sum() / hydro["zone_area_ha"].sum()

stats = {
    "Talar canopy cover (>=2 m)":    f"{canopy.loc['talar', 'cover_2m_pct']:.1f}%",
    "Pastizal canopy cover (>=2 m)": f"{canopy.loc['pastizal', 'cover_2m_pct']:.1f}%",
    "Trees detected (talar)":       f"{n_trees:,}",
    "Ponding-prone area (P>=0.5)":   f"~{ponding_pct:.0f}%",
}

index = wm.write_index_html(
    SITE, map_rel="map.html",
    flood_html_rel="assets/talar_inundation_3d.html",
    gif_rel="assets/talar_flood_rising.gif",
    title="Reserva Natural Municipal del Pilar - Drone Habitat & Flood Atlas",
    subtitle="RGB photogrammetry - structural, habitat & hydrological characterization - Buenos Aires, AR",
    stats=stats,
)
print("wrote", index)
print("stats:", stats)

wrote C:\Users\paco_\OneDrive\Escritorio\repos\drone_reserve\outputs\09_webmap\index.html
stats: {'Talar canopy cover (>=2 m)': '28.8%', 'Pastizal canopy cover (>=2 m)': '20.9%', 'Trees detected (talar)': '1,086', 'Ponding-prone area (P>=0.5)': '~25%'}


## 7. Result

The site is `outputs/09_webmap/`. Open `index.html` locally, or deploy the folder to
GitHub Pages. Preview the map inline below.

In [8]:
total_mb = sum(p.stat().st_size for p in SITE.rglob('*') if p.is_file()) / 1e6
print(f"site total: {total_mb:.0f} MB")
for p in sorted(SITE.iterdir()):
    if p.is_dir():
        nf = sum(1 for _ in p.rglob('*') if _.is_file())
        mb = sum(f.stat().st_size for f in p.rglob('*') if f.is_file()) / 1e6
        print(f"  {p.name}/  ({nf} files, {mb:.1f} MB)")
    else:
        print(f"  {p.name}  ({p.stat().st_size/1024:.0f} KB)")

from IPython.display import IFrame
IFrame(src="../outputs/09_webmap/index.html", width="100%", height=620)

site total: 154 MB
  assets/  (2 files, 4.5 MB)
  data/  (2 files, 0.2 MB)
  index.html  (4 KB)
  map.html  (1157 KB)
  overlay_meta.json  (2 KB)
  overlays/  (8 files, 3.8 MB)


  tiles/  (2113 files, 144.8 MB)
